# Overfit Free Model

In [80]:
# Updated parameters — overfitting fix
train_fraction = 0.8
num_train_epochs = 5          # zyada epochs — early stopping handle karega
batch_size = 32               # bada batch = stable training
warmup_steps = 100            # zyada warmup = smooth start
weight_decay = 0.05           # zyada regularization = overfitting kam
BERT_MODEL = "distilbert-base-uncased"   # uncased = CAPS issue fix
# Updated output directory
output_dir_v2 = "./phishing-email-detection-v2"

In [62]:
# Load new tokenizer
tokenizer_v2 = AutoTokenizer.from_pretrained(BERT_MODEL, use_fast=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [79]:
# Tokenize
def preprocess_function_v2(examples):
    return tokenizer_v2(examples["title"], truncation=True, max_length=512)

medium_dataset_v2 = Dataset.from_pandas(df)
medium_dataset_v2 = medium_dataset_v2.map(preprocess_function_v2, batched=True)
medium_dataset_v2 = medium_dataset_v2.remove_columns("title")
medium_dataset_v2 = medium_dataset_v2.train_test_split(test_size=1-train_fraction)

Map:   0%|          | 0/18634 [00:00<?, ? examples/s]

In [84]:
# Reload dataset
df = pd.read_csv('/kaggle/input/datasets/subhajournal/phishingemails/Phishing_Email.csv')

# Wahi cleaning jo pehle ki thi
df.dropna(inplace=True)

# Check karo
print(df.shape)
print(df.head())

(18634, 3)
   Unnamed: 0                                         Email Text  \
0           0  re : 6 . 1100 , disc : uniformitarianism , re ...   
1           1  the other side of * galicismos * * galicismo *...   
2           2  re : equistar deal tickets are you still avail...   
3           3  \nHello I am your hot lil horny toy.\n    I am...   
4           4  software at incredibly low prices ( 86 % lower...   

       Email Type  
0      Safe Email  
1      Safe Email  
2      Safe Email  
3  Phishing Email  
4  Phishing Email  


In [68]:
# Reload + clean
df = pd.read_csv('/kaggle/input/datasets/subhajournal/phishingemails/Phishing_Email.csv')
df.dropna(inplace=True)

# Column rename karo — purane code se match karne ke liye
df = df.rename(columns={'Email Text': 'title'})

# Label ko 0/1 mein convert karo
df['label'] = df['Email Type'].map({'Safe Email': 0, 'Phishing Email': 1})

# Sirf zaruri columns rakho
df = df[['title', 'label']]

# Check karo
print(df.shape)
print(df['label'].value_counts())
print(df.head())

(18634, 2)
label
0    11322
1     7312
Name: count, dtype: int64
                                               title  label
0  re : 6 . 1100 , disc : uniformitarianism , re ...      0
1  the other side of * galicismos * * galicismo *...      0
2  re : equistar deal tickets are you still avail...      0
3  \nHello I am your hot lil horny toy.\n    I am...      1
4  software at incredibly low prices ( 86 % lower...      1


In [86]:
# Oversampling
y = df[['label']]
df_features = df.drop(['label'], axis=1)
ros = RandomOverSampler(random_state=83)
df_resampled, y_resampled = ros.fit_resample(df_features, y)
df_resampled['label'] = y_resampled
print("After oversampling:", df_resampled.shape)

# HuggingFace Dataset banao
medium_dataset_v2 = Dataset.from_pandas(df_resampled)

# Tokenize
def preprocess_function_v2(examples):
    return tokenizer_v2(examples["title"], truncation=True, max_length=512)

medium_dataset_v2 = medium_dataset_v2.map(preprocess_function_v2, batched=True)
medium_dataset_v2 = medium_dataset_v2.remove_columns("title")
medium_dataset_v2 = medium_dataset_v2.train_test_split(test_size=1-train_fraction)

print(medium_dataset_v2)

KeyError: "None of [Index(['label'], dtype='object')] are in the [columns]"

In [ ]:
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL,
    num_labels=2,
    seq_classif_dropout=0.3,    # classifier layer dropout ✅
    dropout=0.2,                 # hidden layers dropout ✅
    output_attentions=False,
    output_hidden_states=False
)
model_v2.config.id2label = {0: 'SAVE EMAIL', 1: 'PHISHING EMAIL'}

In [72]:
data_collator_v2 = DataCollatorWithPadding(tokenizer=tokenizer_v2)

In [73]:
metric_v2 = evaluate.load("accuracy")

def compute_metrics_v2(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric_v2.compute(predictions=predictions, references=labels)

In [81]:
training_args_v2 = TrainingArguments(
    output_dir=output_dir_v2,        # ← v2 folder
    logging_dir='./logs_v2',
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    logging_strategy='steps',
    logging_first_step=True,
    load_best_model_at_end=True,
    logging_steps=10,
    eval_strategy='epoch',
    warmup_steps=warmup_steps,
    weight_decay=weight_decay,
    save_strategy='epoch',
    report_to="mlflow",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [82]:
trainer_v2 = Trainer(
    model=model_v2,
    args=training_args_v2,
    compute_metrics=compute_metrics_v2,
    train_dataset=medium_dataset_v2['train'],
    eval_dataset=medium_dataset_v2['test'],
    data_collator=data_collator_v2,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [87]:
# Baseline
trainer_v2.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.3719247579574585,
 'eval_accuracy': 0.6168500134156157,
 'eval_runtime': 32.9044,
 'eval_samples_per_second': 113.267,
 'eval_steps_per_second': 1.793}

In [88]:
trainer_v2.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.199682,0.189277,0.965388
2,0.096676,0.107588,0.982023
3,0.053727,0.115829,0.981218
4,0.046319,0.128263,0.982291


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=932, training_loss=0.15691929020771014, metrics={'train_runtime': 1633.4431, 'train_samples_per_second': 45.631, 'train_steps_per_second': 0.713, 'total_flos': 7898766047059968.0, 'train_loss': 0.15691929020771014, 'epoch': 4.0})

In [89]:
# Final evaluation
trainer_v2.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.10763367265462875,
 'eval_accuracy': 0.982023074859136,
 'eval_runtime': 30.1856,
 'eval_samples_per_second': 123.469,
 'eval_steps_per_second': 1.955,
 'epoch': 4.0}

In [90]:
# Save
trainer_v2.save_model()
tokenizer_v2.save_pretrained(output_dir_v2)
print("Model saved to:", output_dir_v2)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./phishing-email-detection-v2


In [91]:
# Test karo
pipe_v2 = pipeline("text-classification", output_dir_v2, tokenizer=tokenizer_v2)

# Normal email
sample1 = "Why do employees leave companies — analysis of IBM employee data"
print("Normal:", pipe_v2(sample1, top_k=None))

# CAPS test
sample2 = sample1.upper()
print("CAPS:", pipe_v2(sample2, top_k=None))

# Phishing email
sample3 = "URGENT! Click here to claim your FREE prize now!"
print("Phishing:", pipe_v2(sample3, top_k=None))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Normal: [{'label': 'SAVE EMAIL', 'score': 0.737289309501648}, {'label': 'PHISHING EMAIL', 'score': 0.26271066069602966}]
CAPS: [{'label': 'SAVE EMAIL', 'score': 0.737289309501648}, {'label': 'PHISHING EMAIL', 'score': 0.26271066069602966}]
Phishing: [{'label': 'PHISHING EMAIL', 'score': 0.9991430044174194}, {'label': 'SAVE EMAIL', 'score': 0.0008570231730118394}]


In [92]:
# More testing
test_emails = [
    # Clear phishing
    "WINNER! You have been selected for a $1000 gift card! Click NOW to claim!",
    "Your account has been compromised! Verify your password immediately!",
    "Congratulations! You won a FREE iPhone! Limited time offer, act fast!",
    
    # Clear safe
    "Please find attached the quarterly financial report for Q2 2026.",
    "Hi team, let's schedule a meeting for Monday at 10am to discuss the project.",
    "Your order #12345 has been shipped and will arrive by Friday.",
    
    # Tricky ones
    "Your invoice is due. Please click here to make a payment.",
    "We need to verify your account details for security purposes.",
    "Dear customer, your subscription will expire soon. Renew now!",
    
    # CAPS mix
    "MEETING SCHEDULED FOR MONDAY AT 10AM IN CONFERENCE ROOM B",
    "FREE WEBINAR: Machine Learning in 2026 — Register Now!",
]

print("=" * 60)
for email in test_emails:
    result = pipe_v2(email, top_k=None)
    top = max(result, key=lambda x: x['score'])
    confidence = top['score'] * 100
    label = top['label']
    icon = "🔴" if label == "PHISHING EMAIL" else "🟢"
    print(f"{icon} {label} ({confidence:.1f}%)")
    print(f"   📧 {email[:55]}...")
    print()

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


🔴 PHISHING EMAIL (99.9%)
   📧 WINNER! You have been selected for a $1000 gift card! C...

🔴 PHISHING EMAIL (99.9%)
   📧 Your account has been compromised! Verify your password...

🔴 PHISHING EMAIL (99.9%)
   📧 Congratulations! You won a FREE iPhone! Limited time of...

🟢 SAVE EMAIL (99.9%)
   📧 Please find attached the quarterly financial report for...

🟢 SAVE EMAIL (100.0%)
   📧 Hi team, let's schedule a meeting for Monday at 10am to...

🟢 SAVE EMAIL (58.5%)
   📧 Your order #12345 has been shipped and will arrive by F...

🔴 PHISHING EMAIL (99.8%)
   📧 Your invoice is due. Please click here to make a paymen...

🔴 PHISHING EMAIL (99.5%)
   📧 We need to verify your account details for security pur...

🔴 PHISHING EMAIL (99.8%)
   📧 Dear customer, your subscription will expire soon. Rene...

🟢 SAVE EMAIL (99.9%)
   📧 MEETING SCHEDULED FOR MONDAY AT 10AM IN CONFERENCE ROOM...

🔴 PHISHING EMAIL (99.9%)
   📧 FREE WEBINAR: Machine Learning in 2026 — Register Now!...



In [95]:
import os
files = os.listdir('./phishing-email-detection-v2')
for f in files:
    size = os.path.getsize(f'./phishing-email-detection-v2/{f}') / (1024*1024)
    print(f"{f} — {size:.1f} MB")

checkpoint-466 — 0.0 MB
config.json — 0.0 MB
tokenizer_config.json — 0.0 MB
training_args.bin — 0.0 MB
checkpoint-932 — 0.0 MB
model.safetensors — 255.4 MB
checkpoint-699 — 0.0 MB
checkpoint-233 — 0.0 MB
tokenizer.json — 0.7 MB


In [94]:
# Missing files save karo
tokenizer_v2.save_pretrained('./phishing-email-detection-v2')
print("Done!")
print(os.listdir('./phishing-email-detection-v2'))

Done!
['checkpoint-466', 'config.json', 'tokenizer_config.json', 'training_args.bin', 'checkpoint-932', 'model.safetensors', 'checkpoint-699', 'checkpoint-233', 'tokenizer.json']


In [101]:
import shutil
shutil.make_archive('phishing-model-v2', 'zip', './phishing-email-detection-v2')
print("ZIP ready!")

ZIP ready!


In [99]:
tokenizer_v2.save_pretrained('./phishing-email-detection-v2')
print(os.listdir('./phishing-email-detection-v2'))

['config.json', 'tokenizer_config.json', 'training_args.bin', 'model.safetensors', 'tokenizer.json']


In [100]:
import json

special_tokens = {
    "unk_token": "[UNK]",
    "sep_token": "[SEP]",
    "pad_token": "[PAD]",
    "cls_token": "[CLS]",
    "mask_token": "[MASK]"
}

with open('./phishing-email-detection-v2/special_tokens_map.json', 'w') as f:
    json.dump(special_tokens, f, indent=2)

print("Done!")
print(os.listdir('./phishing-email-detection-v2'))

Done!
['config.json', 'tokenizer_config.json', 'training_args.bin', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json']


In [102]:
from IPython.display import FileLink
FileLink('phishing-model-v2.zip')

/kaggle/working/phishing-model-v2.zip